In [2]:
# # Pull the SynthStrip Docker image (run once)
# import subprocess
# result = subprocess.run(["docker", "pull", "freesurfer/synthstrip"], capture_output=False)
# print("Docker pull complete" if result.returncode == 0 else "Docker pull failed — is Docker Desktop running?")

In [4]:
# !pip install antspyx nibabel numpy matplotlib tqdm templateflow

In [5]:
from templateflow import api as tflow

ATLAS_COHORT = "3"   # 4.5–8.5 years — change to match your subjects

template_t1 = tflow.get(
    "MNIPediatricAsym", cohort=ATLAS_COHORT, resolution=1,
    suffix="T1w", extension=".nii.gz",
)
template_mask = tflow.get(
    "MNIPediatricAsym", cohort=ATLAS_COHORT, resolution=1,
    suffix="mask", label="brain", extension=".nii.gz",
)
print(f"Template T1  : {template_t1}")
print(f"Template mask: {template_mask}")

100%|██████████████████████████████████████████████████████████████████████████████| 4.40M/4.40M [00:05<00:00, 836kB/s]

Template T1  : C:\Users\DELL\AppData\Local\templateflow\templateflow\Cache\tpl-MNIPediatricAsym\cohort-3\tpl-MNIPediatricAsym_cohort-3_res-1_T1w.nii.gz
Template mask: []


## A2. Imports & Configuration

In [21]:
import os
import subprocess
import shutil
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm

# ── Paths ────────────────────────────────────────────────────
DATA_ROOT   = Path(r"C:\Users\DELL\Desktop\Sanskriti\Brats_pred\All_data_raw\Test")           # Input dataset root
OUTPUT_ROOT = Path(r"C:\Users\DELL\Desktop\Sanskriti\Brats_pred\All_data_Deskullp\Test")  # Output root (no spaces!)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# BraTS-PEDs modality suffixes
MODALITIES = {
    "t1n": "-t1n.nii.gz",
    "t1c": "-t1c.nii.gz",
    "t2w": "-t2w.nii.gz",
    "t2f": "-t2f.nii.gz",
}
MASK_MODALITY = "t1c"   # modality used to generate the brain mask

# SynthStrip settings
USE_DOCKER   = True     # True = Docker; False = FreeSurfer CLI (mri_synthstrip)
NO_CSF       = False    # True = exclude CSF from brain mask (tighter mask)
BORDER       = 1        # Mask border dilation in mm (default 1)

print(f"Data root    : {DATA_ROOT}")
print(f"Output root  : {OUTPUT_ROOT}")
print(f"Mask modality: {MASK_MODALITY}")
print(f"Use Docker   : {USE_DOCKER}")

Data root    : C:\Users\DELL\Desktop\Sanskriti\Brats_pred\All_data_raw\Test
Output root  : C:\Users\DELL\Desktop\Sanskriti\Brats_pred\All_data_Deskullp\Test
Mask modality: t1c
Use Docker   : True


## A3. SynthStrip Helper Functions

In [22]:
def run_synthstrip_docker(
    input_path: Path,
    output_path: Path,
    mask_path: Path,
    no_csf: bool = False,
    border: int = 1,
) -> bool:
    """
    Run SynthStrip via Docker.
    Mounts only the subject folder — avoids long path issues on Windows.
    """
    # Docker needs Linux-style paths; use PurePosixPath logic via string replace
    host_dir  = input_path.parent
    cont_dir  = "/data"
    cont_in   = f"{cont_dir}/{input_path.name}"
    cont_out  = f"{cont_dir}/{output_path.name}"
    cont_mask = f"{cont_dir}/{mask_path.name}"

    output_path.parent.mkdir(parents=True, exist_ok=True)

    cmd = [
        "docker", "run", "--rm",
        "-v", f"{host_dir}:{cont_dir}",
        "freesurfer/synthstrip",
        "-i", cont_in,
        "-o", cont_out,
        "-m", cont_mask,
        "--border", str(border),
    ]
    if no_csf:
        cmd.append("--no-csf")

    # Copy input to host_dir if output_path.parent differs from input_path.parent
    tmp_input = None
    if output_path.parent != input_path.parent:
        tmp_input = output_path.parent / input_path.name
        shutil.copy2(input_path, tmp_input)
        cmd[cmd.index(f"{host_dir}:{cont_dir}")]  = f"{output_path.parent}:{cont_dir}"
        cmd[cmd.index(cont_in)]  = f"{cont_dir}/{input_path.name}"

    print(f"  CMD: {' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=False)

    # Clean up temp copy
    if tmp_input and tmp_input.exists():
        tmp_input.unlink()

    if result.returncode != 0:
        print(f"  [ERROR] SynthStrip Docker failed (code {result.returncode})")
        return False
    if not output_path.exists():
        print(f"  [ERROR] Output not created: {output_path}")
        return False

    print(f"  [OK] Brain: {output_path.name}")
    print(f"  [OK] Mask : {mask_path.name}")
    return True


def run_synthstrip_freesurfer(
    input_path: Path,
    output_path: Path,
    mask_path: Path,
    no_csf: bool = False,
    border: int = 1,
) -> bool:
    """
    Run SynthStrip via FreeSurfer CLI (mri_synthstrip).
    Requires FreeSurfer to be installed and FREESURFER_HOME set.
    """
    output_path.parent.mkdir(parents=True, exist_ok=True)
    cmd = [
        "mri_synthstrip",
        "-i", str(input_path),
        "-o", str(output_path),
        "-m", str(mask_path),
        "--border", str(border),
    ]
    if no_csf:
        cmd.append("--no-csf")

    print(f"  CMD: {' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=False)

    if result.returncode != 0 or not output_path.exists():
        print(f"  [ERROR] mri_synthstrip failed")
        return False
    print(f"  [OK] Brain: {output_path.name}")
    return True


def apply_mask(input_path: Path, mask_path: Path, output_path: Path) -> None:
    """Apply binary brain mask to a NIfTI volume."""
    img  = nib.load(input_path)
    mask = nib.load(mask_path)
    stripped = img.get_fdata() * mask.get_fdata().astype(bool)
    out = nib.Nifti1Image(stripped, img.affine, img.header)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    nib.save(out, str(output_path))
    print(f"  [OK] Mask applied → {output_path.name}")


print("SynthStrip helper functions defined.")

SynthStrip helper functions defined.


## A4. Single-Subject Test

In [23]:
# Get subjects
subjects = sorted([d for d in DATA_ROOT.iterdir() if d.is_dir()])
print(f"Found {len(subjects)} subjects")

test_subject = subjects[0]
subject_id   = test_subject.name
out_dir      = OUTPUT_ROOT / subject_id
out_dir.mkdir(parents=True, exist_ok=True)

input_file = test_subject / f"{subject_id}{MODALITIES[MASK_MODALITY]}"
out_file   = out_dir / f"{subject_id}_{MASK_MODALITY}_synthstrip.nii.gz"
mask_file  = out_dir / f"{subject_id}_{MASK_MODALITY}_mask.nii.gz"

print(f"Subject : {subject_id}")
print(f"Input   : {input_file}")
print(f"Output  : {out_file}")
print(f"Mask    : {mask_file}")

if USE_DOCKER:
    success = run_synthstrip_docker(
        input_path=input_file, output_path=out_file, mask_path=mask_file,
        no_csf=NO_CSF, border=BORDER,
    )
else:
    success = run_synthstrip_freesurfer(
        input_path=input_file, output_path=out_file, mask_path=mask_file,
        no_csf=NO_CSF, border=BORDER,
    )

print("\nTest PASSED" if success else "\nTest FAILED")

Found 20 subjects
Subject : BraTS-PED-00010-000
Input   : C:\Users\DELL\Desktop\Sanskriti\Brats_pred\All_data_raw\Test\BraTS-PED-00010-000\BraTS-PED-00010-000-t1c.nii.gz
Output  : C:\Users\DELL\Desktop\Sanskriti\Brats_pred\All_data_Deskullp\Test\BraTS-PED-00010-000\BraTS-PED-00010-000_t1c_synthstrip.nii.gz
Mask    : C:\Users\DELL\Desktop\Sanskriti\Brats_pred\All_data_Deskullp\Test\BraTS-PED-00010-000\BraTS-PED-00010-000_t1c_mask.nii.gz
  CMD: docker run --rm -v C:\Users\DELL\Desktop\Sanskriti\Brats_pred\All_data_Deskullp\Test\BraTS-PED-00010-000:/data freesurfer/synthstrip -i /data/BraTS-PED-00010-000-t1c.nii.gz -o /data/BraTS-PED-00010-000_t1c_synthstrip.nii.gz -m /data/BraTS-PED-00010-000_t1c_mask.nii.gz --border 1
  [OK] Brain: BraTS-PED-00010-000_t1c_synthstrip.nii.gz
  [OK] Mask : BraTS-PED-00010-000_t1c_mask.nii.gz

Test PASSED


## A5. Apply Mask to All Modalities

In [24]:
subject_id = subjects[0].name
out_dir    = OUTPUT_ROOT / subject_id
mask_file  = out_dir / f"{subject_id}_{MASK_MODALITY}_mask.nii.gz"

if not mask_file.exists():
    print(f"[ERROR] Mask not found: {mask_file}\nRun Cell A4 first.")
else:
    for mod_key, mod_suffix in MODALITIES.items():
        if mod_key == MASK_MODALITY:
            continue
        inp = subjects[0] / f"{subject_id}{mod_suffix}"
        out = out_dir / f"{subject_id}_{mod_key}_synthstrip.nii.gz"
        if inp.exists():
            apply_mask(inp, mask_file, out)
        else:
            print(f"  [SKIP] {inp.name} not found")
    print("\nDone — all modalities skull-stripped.")

  [OK] Mask applied → BraTS-PED-00010-000_t1n_synthstrip.nii.gz
  [OK] Mask applied → BraTS-PED-00010-000_t2w_synthstrip.nii.gz
  [OK] Mask applied → BraTS-PED-00010-000_t2f_synthstrip.nii.gz

Done — all modalities skull-stripped.


## A6. Full Batch

In [25]:
failed = []

for subject_dir in tqdm(subjects, desc="SynthStrip batch"):
    sid     = subject_dir.name
    out_dir = OUTPUT_ROOT / sid
    out_dir.mkdir(parents=True, exist_ok=True)

    inp       = subject_dir / f"{sid}{MODALITIES[MASK_MODALITY]}"
    out_brain = out_dir / f"{sid}_{MASK_MODALITY}_synthstrip.nii.gz"
    out_mask  = out_dir / f"{sid}_{MASK_MODALITY}_mask.nii.gz"

    if not inp.exists():
        print(f"  [SKIP] {sid}: input missing")
        failed.append(sid)
        continue

    fn = run_synthstrip_docker if USE_DOCKER else run_synthstrip_freesurfer
    ok = fn(inp, out_brain, out_mask, no_csf=NO_CSF, border=BORDER)

    if ok:
        for mod_key, mod_suffix in MODALITIES.items():
            if mod_key == MASK_MODALITY:
                continue
            mod_inp = subject_dir / f"{sid}{mod_suffix}"
            mod_out = out_dir / f"{sid}_{mod_key}_synthstrip.nii.gz"
            if mod_inp.exists():
                apply_mask(mod_inp, out_mask, mod_out)
    else:
        failed.append(sid)

print(f"\nBatch done: {len(subjects)-len(failed)}/{len(subjects)} succeeded")
if failed:
    print("Failed:", failed)

SynthStrip batch:   0%|                                                                         | 0/20 [00:00<?, ?it/s]

  CMD: docker run --rm -v C:\Users\DELL\Desktop\Sanskriti\Brats_pred\All_data_Deskullp\Test\BraTS-PED-00010-000:/data freesurfer/synthstrip -i /data/BraTS-PED-00010-000-t1c.nii.gz -o /data/BraTS-PED-00010-000_t1c_synthstrip.nii.gz -m /data/BraTS-PED-00010-000_t1c_mask.nii.gz --border 1
  [OK] Brain: BraTS-PED-00010-000_t1c_synthstrip.nii.gz
  [OK] Mask : BraTS-PED-00010-000_t1c_mask.nii.gz
  [OK] Mask applied → BraTS-PED-00010-000_t1n_synthstrip.nii.gz
  [OK] Mask applied → BraTS-PED-00010-000_t2w_synthstrip.nii.gz


SynthStrip batch:   5%|███▎                                                             | 1/20 [00:16<05:18, 16.74s/it]

  [OK] Mask applied → BraTS-PED-00010-000_t2f_synthstrip.nii.gz
  CMD: docker run --rm -v C:\Users\DELL\Desktop\Sanskriti\Brats_pred\All_data_Deskullp\Test\BraTS-PED-00013-000:/data freesurfer/synthstrip -i /data/BraTS-PED-00013-000-t1c.nii.gz -o /data/BraTS-PED-00013-000_t1c_synthstrip.nii.gz -m /data/BraTS-PED-00013-000_t1c_mask.nii.gz --border 1
  [OK] Brain: BraTS-PED-00013-000_t1c_synthstrip.nii.gz
  [OK] Mask : BraTS-PED-00013-000_t1c_mask.nii.gz
  [OK] Mask applied → BraTS-PED-00013-000_t1n_synthstrip.nii.gz
  [OK] Mask applied → BraTS-PED-00013-000_t2w_synthstrip.nii.gz


SynthStrip batch:  10%|██████▌                                                          | 2/20 [00:35<05:27, 18.18s/it]

...

  [OK] Mask applied → BraTS-PED-00030-000_t2f_synthstrip.nii.gz
  CMD: docker run --rm -v C:\Users\DELL\Desktop\Sanskriti\Brats_pred\All_data_Deskullp\Test\BraTS-PED-00031-000:/data freesurfer/synthstrip -i /data/BraTS-PED-00031-000-t1c.nii.gz -o /data/BraTS-PED-00031-000_t1c_synthstrip.nii.gz -m /data/BraTS-PED-00031-000_t1c_mask.nii.gz --border 1
  [OK] Brain: BraTS-PED-00031-000_t1c_synthstrip.nii.gz
  [OK] Mask : BraTS-PED-00031-000_t1c_mask.nii.gz
  [OK] Mask applied → BraTS-PED-00031-000_t1n_synthstrip.nii.gz
  [OK] Mask applied → BraTS-PED-00031-000_t2w_synthstrip.nii.gz


SynthStrip batch:  95%|████████████████████████████████████████████████████████████▊   | 19/20 [05:22<00:17, 17.13s/it]

  [OK] Mask applied → BraTS-PED-00031-000_t2f_synthstrip.nii.gz
  CMD: docker run --rm -v C:\Users\DELL\Desktop\Sanskriti\Brats_pred\All_data_Deskullp\Test\BraTS-PED-00032-000:/data freesurfer/synthstrip -i /data/BraTS-PED-00032-000-t1c.nii.gz -o /data/BraTS-PED-00032-000_t1c_synthstrip.nii.gz -m /data/BraTS-PED-00032-000_t1c_mask.nii.gz --border 1
  [OK] Brain: BraTS-PED-00032-000_t1c_synthstrip.nii.gz
  [OK] Mask : BraTS-PED-00032-000_t1c_mask.nii.gz
  [OK] Mask applied → BraTS-PED-00032-000_t1n_synthstrip.nii.gz
  [OK] Mask applied → BraTS-PED-00032-000_t2w_synthstrip.nii.gz


SynthStrip batch: 100%|████████████████████████████████████████████████████████████████| 20/20 [05:36<00:00, 16.84s/it]

  [OK] Mask applied → BraTS-PED-00032-000_t2f_synthstrip.nii.gz

Batch done: 20/20 succeeded


## Save the mask in the same folder

In [ ]:
import shutil
from pathlib import Path

# ── Config ───────────────────────────────────────────────────
DATA_ROOT   = Path(r"C:\Users\DELL\Desktop\Sanskriti\Brats_pred\All_data_raw\Test")           # original BraTS data
OUTPUT_ROOT = Path(r"C:\Users\DELL\Desktop\Sanskriti\Brats_pred\All_data_Deskullp\Test")  # skull-stripped output

# BraTS-PEDs segmentation mask suffix — adjust if yours differs
SEG_SUFFIX = "-seg.nii.gz"   # e.g. BraTS-PED-00013-101-seg.nii.gz

# ── Copy seg masks into each subject's skull-stripped folder ─
subjects_out = sorted([d for d in OUTPUT_ROOT.iterdir() if d.is_dir()])
print(f"Found {len(subjects_out)} subjects in output folder\n")

copied = 0
missed = 0

for out_dir in subjects_out:
    sid = out_dir.name

    # Source: original BraTS seg file
    seg_src = DATA_ROOT / sid / f"{sid}{SEG_SUFFIX}"

    # Destination: inside the skull-stripped subject folder
    seg_dst = out_dir / f"{sid}{SEG_SUFFIX}"

    if seg_src.exists():
        shutil.copy2(seg_src, seg_dst)
        print(f"  [OK]   {sid} → {seg_dst.name}")
        copied += 1
    else:
        # Try to find any seg file automatically
        found = list((DATA_ROOT / sid).glob("*seg*"))
        if found:
            seg_dst = out_dir / found[0].name
            shutil.copy2(found[0], seg_dst)
            print(f"  [OK]   {sid} → {found[0].name} (auto-found)")
            copied += 1
        else:
            print(f"  [MISS] {sid}: seg file not found in {DATA_ROOT / sid}")
            missed += 1

print(f"\nDone — {copied} seg masks copied, {missed} missing")